In [1]:
DIVIDING = '-' * 100
chr(0)
print(1)
print(chr(0))
print(1)
print('-'*100)
print(chr(0).__repr__())
#'\x00'在文本中表示换行符？

1
 
1
----------------------------------------------------------------------------------------------------
'\x00'


In [2]:
test_string = "hello! こんにちは!"
utf8_enconded = test_string.encode('utf-8')
print(utf8_enconded)
print(DIVIDING)
print(type(utf8_enconded))
print(DIVIDING)
print(list(utf8_enconded))
print(DIVIDING)
print(len(test_string))
print(DIVIDING)
print(len(utf8_enconded))
print(DIVIDING)
print(utf8_enconded.decode('utf-8'))
print('utf-16:',test_string.encode('utf-16'),len(test_string.encode('utf-16')))
print('utf-32:',test_string.encode('utf-32'),len(test_string.encode('utf-32')))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
----------------------------------------------------------------------------------------------------
<class 'bytes'>
----------------------------------------------------------------------------------------------------
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
----------------------------------------------------------------------------------------------------
13
----------------------------------------------------------------------------------------------------
23
----------------------------------------------------------------------------------------------------
hello! こんにちは!
utf-16: b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00S0\x930k0a0o0!\x00' 28
utf-32: b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00!\x00\x00\x00' 56


utf16,utf32表示字符串时，会出现一些如\x00的字符，浪费了表示空间。

In [4]:
def decode_utf8_bytes_to_str_wrong(bytestring):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
#decode_utf8_bytes_to_str_wrong("你hello are you a girl".encode("utf-8"))
#utf8是多字节编码，ASCII字符占一个字节，中文占2字节，emoji占4字节，不能分开解码
error_byte_string = b'\xc2\x88'
print(error_byte_string.decode('utf-8'))
#UTF-8 中，两字节序列的合法格式是 110xxxxx 10xxxxxx，即第一个字节必须在 0xC2 到 0xDF 之间，第二个字节必须在 0x80 到 0xBF 之间。




In [15]:
def vocab_init():
    return {i:bytes([i]) for i in range(256)}
vocab = vocab_init()
print(vocab)
print(vocab[0])
word = 'hello'
print(word.encode('utf-8'))
print(list(word.encode('utf-8')))
t = (101,108)
t = list(t)
print(t)
print(bytes(t).decode())
#print(t.decode)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [ ]:
from collections import Counter
import heapq
import regex
def vocab_init():
    return {i:bytes([i]) for i in range(256)}
def train_bpe(text,merge_time):
    #O(M*N*L)
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    vocab = vocab_init()
    heap = []
    freqs = {}
    words = {}
    merges = {}
    itw = {}
    ptw = {}
    w_id = 0
    for m in regex.finditer(PAT,text):
        ids = tuple(list(m.group().encode('utf-8')))
        if ids not in itw.keys():
            itw[ids] = w_id
            w_id += 1
        id = itw[ids]
        words[id] = (words.get(id,(0,ids))[0] + 1, ids)
    for k,v in words.items():
        for pair in zip(v[1],v[1][1:]):
            freqs[pair] = freqs.get(pair,0) + v[0]
            ptw.setdefault(pair,set()).add(k)
    for pair,count in freqs.items():
        heapq.heappush(heap,(-count,pair))
    max_key = None
    for iter in range(merge_time):
        idx = 256 + iter
        #不重新创建freqs和ptw，而是只维护增量
        #M个单词，单词平均长度为N O(M*N)
        if not freqs:
            break
        # max_key = max(freqs,key=freqs.get)
        while heap:
            neg_cnt,pair = heap[0]
            real_cnt = freqs.get(pair,0)
            if real_cnt == 0 or -neg_cnt != real_cnt:
                heapq.heappop(heap)
            else:
                max_key = pair
                break
        if max_key == None:
            break
        affected = ptw.pop(max_key,set())
        for word_id in list(affected):
        #每个word都是受到影响的单词，要维护ptw：1.删除[max_key]键值对 2.把所有值列表中的受影响单词(包含max_key的)用新词替换(只能遍历？)
        #words = word_id ->(ids,freq)， ptw = pair -> [word_id]
        #把words中受影响的单词去掉,添加新单词，在ptw中找到受影响的序号，用新的换掉
            i = 0
            new_word = []
            freq,word= words.pop(word_id)
            old_pairs = Counter(zip(word,word[1:]))
            while i < len(word):
                if i < len(word) - 1 and word[i] == max_key[0] and word[i+1] == max_key[1]:
                    new_word.append(idx)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_pairs = Counter(zip(new_word,new_word[1:]))
            new_word = tuple(new_word)
            for p,c in old_pairs.items():
                freqs[p] -= freq * c
                if freqs[p] <= 0:
                    freqs.pop(p)
                s = ptw.get(p)
                if s is not None:
                    s.discard(word_id)
                    if not s:
                        del ptw[p]
            for p,c in new_pairs.items():
                freqs[p] = freqs.get(p,0) + freq * c
                ptw.setdefault(p,set()).add(word_id)
                heapq.heappush(heap,(-freqs[p],p))
            words[word_id] = (freq, new_word)
        merges[idx] = max_key
        vocab[idx] = vocab[max_key[0]] + vocab[max_key[1]]
    return merges,vocab

merges, vocab = train_bpe("low low low low low lower lower widest widest widest newest newest newest newest newest",6)
print(merges)
print(vocab)

{256: (101, 115), 257: (108, 111), 258: (257, 119), 259: (32, 258), 260: (32, 110), 261: (256, 116)}
{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q'

word = (1,2,3)
print(word[1:])

In [12]:
word = 'hello world'
ids = word.encode('utf-8')
print(list(zip(ids,ids[1:])))

[(104, 101), (101, 108), (108, 108), (108, 111), (111, 32), (32, 119), (119, 111), (111, 114), (114, 108), (108, 100)]


In [9]:
merges = []
pair = (bytes([110]),bytes([110]))
print(pair)
merges.append(pair)
print(merges)

(b'n', b'n')
[(b'n', b'n')]


In [2]:
def bytes_to_unicode():
    """
    返回一个映射：0~255 的每个字节 -> 一个可见的 unicode 字符。
    可逆，用于把任意 bytes token 安全地放进 JSON / 字符串。
    """
    # 可见 ASCII 段 + 拉丁补充段，直接一一对应
    bs = (
        list(range(ord("!"), ord("~") + 1))      # 33..126
        + list(range(ord("¡"), ord("¬") + 1))    # 161..172
        + list(range(ord("®"), ord("ÿ") + 1))    # 174..255
    )
    cs = bs[:]
    n = 0
    # 剩下的字节（控制字符、0x80~0xA0 等）映射到 256, 257, ... 对应的字符
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}

btu = bytes_to_unicode()
words = b'eiwopa'
res = ''
for c in words:
    res += btu[c]
print(res)

eiwopa


In [3]:
words = 'abcds'
words = words.encode('utf-8')
print(bytes(words))

b'abcds'
